# 11 — Reflection-память: цикл реально учится

> **Главный исследовательский вопрос ТЗ:** как сделать так, чтобы
> генератор учился на замечаниях судьи, а не повторял те же ошибки?
>
> **Решение:** между итерациями вставляем `reflector` — отдельный
> узел, который переписывает findings в **компактные `Lesson`**, и
> они подкладываются в системный промпт generator'а на следующей
> итерации.

## Что мы покажем

1. Mock-генератор: возвращает SQL по правилу «без reflection → повторяет ошибку».
2. Mock-судья: упрощённый Phase 1 на 9 правилах из ADR-0004.
3. Reflector: findings → `Lesson` (список 5 последних, deduped).
4. Цикл `generator → judge → (reflector) → generator → ...` до 5 итераций.
5. A/B: **с reflection vs без**. Видим, что без reflection цикл не сходится.


## 🧒 Аналогия для ребёнка

Ты решаешь контрольную по математике. Учитель проверяет, ставит
тебе **галочки и крестики** на ошибки.

- **Без reflection:** ты пересдаёшь и **снова делаешь те же ошибки**,
  потому что забыл что было не так. Учитель снова ставит крестики.
  Это бесконечно.
- **С reflection:** перед пересдачей ты **записываешь
  в шпаргалку**: «не путать минус и плюс при переносе через =».
  Берёшь шпаргалку на пересдачу — больше эту ошибку не делаешь.

В нашем цикле: shpargalka = `Lesson`. Reflector — это ты, кто
пишет шпаргалку по галочкам учителя. Generator — снова пишущий
контрольную, но теперь с шпаргалкой в руке.


## 1. Setup — mock-генератор, mock-судья, состояние цикла


In [ ]:
"""
@brief Подготовка окружения и mock-БД через in-memory SQLite.
@details
    Никаких внешних зависимостей кроме stdlib + sqlite3 (есть в Colab из коробки).
    SQLite используем как «упрощённую модель PostgreSQL» — он умеет
    почти весь стандартный SQL, что достаточно для демонстраций уязвимостей.
@note
    Реальная система работает на PostgreSQL (см. ADR-0001),
    использует pglast для AST-парсинга. Здесь, для наглядности,
    эмулируем аудитор через `re` (регулярки) и простой pattern matching.
"""
import sqlite3
import re
import time
from textwrap import dedent


def section(title):
    """@brief Печатает заголовок секции."""
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)


def show_result(rows, max_rows=10):
    """@brief Печатает результаты запроса в виде таблицы."""
    if not rows:
        print("  (нет строк)")
        return
    for i, r in enumerate(rows[:max_rows]):
        print(f"  {i + 1:>3}. {r}")
    if len(rows) > max_rows:
        print(f"  ... ещё {len(rows) - max_rows} строк")


def print_finding(f):
    """@brief Красиво печатает Finding от нашего аудитора."""
    print(f"  ⚠️  {f['rule_id']}")
    print(f"      vuln_class:  {f['vuln_class']}")
    print(f"      severity:    {f['severity']}")
    print(f"      risk_score:  {f['risk_score']}/10")
    print(f"      message:     {f['message']}")
    if f.get("evidence_refs"):
        print(f"      ссылки:      {', '.join(f['evidence_refs'])}")


from dataclasses import dataclass, field
from typing import Any


##
# @brief Имитация state-объекта LangGraph (см. ADR-0002).
@dataclass
class LoopState:
    task: str                              #!< NL-вопрос
    sql_history: list = field(default_factory=list)
    audit_history: list = field(default_factory=list)
    reflection: list = field(default_factory=list)  #!< последние Lesson-ы
    iteration: int = 0
    approved: bool = False
    final_sql: str = ""


##
# @brief Один lesson, который reflector кладёт в state.
@dataclass
class Lesson:
    rule_id: str
    lesson: str
    example_bad: str
    example_good: str

    def __str__(self):
        return f"[{self.rule_id}] {self.lesson}"


##
# @brief Mock-генератор. Эмулирует поведение Qwen-Coder.
# @param task         NL-вопрос.
# @param reflection   Накопленные lesson-ы из state.
# @return             SQL-строка.
# @details
#   Здесь мы симулируем «эволюцию ответа модели»:
#   - на iter 1 модель выдаёт SQL с тремя ошибками: SELECT *, без WHERE, без LIMIT;
#   - на каждой следующей итерации, если в reflection есть lesson по правилу,
#     модель «исправляет» соответствующую ошибку. Без reflection — повторяет.
def mock_generator(task, reflection):
    rules_to_fix = {l.rule_id for l in reflection}
    # Базовая «плохая» версия
    select_part = "id, full_name" if "DIRECT_SENSITIVE" in rules_to_fix or "SELECT_STAR" in rules_to_fix else "*"
    where_part = "WHERE balance > 0" if "DML_NO_WHERE" in rules_to_fix else ""
    limit_part = "LIMIT 100" if "NO_PAGINATION" in rules_to_fix else ""
    return f"SELECT {select_part} FROM clients {where_part} {limit_part}".strip()


section("Тест mock_generator: пустая reflection")
print(f"  iter1 SQL: {mock_generator('покажи клиентов', [])}")

section("Тест mock_generator: с lesson по SELECT_STAR")
lessons = [Lesson("SELECT_STAR", "не используй *", "SELECT * FROM ...", "SELECT id, full_name FROM ...")]
print(f"  iter2 SQL: {mock_generator('покажи клиентов', lessons)}")


## 2. Mock-судья (Phase 1 правила)


In [ ]:
##
# @brief Mock-судья: симулирует Phase 1 правила.
def mock_judge(sql):
    findings = []
    if re.search(r"SELECT\s+\*", sql, re.IGNORECASE):
        findings.append({"rule_id": "SELECT_STAR", "risk_score": 5,
                         "message": "SELECT * — раскрывает все колонки"})
    if re.search(r"\bDELETE\b|\bUPDATE\b", sql, re.IGNORECASE) and \
       not re.search(r"\bWHERE\b", sql, re.IGNORECASE):
        findings.append({"rule_id": "DML_NO_WHERE", "risk_score": 9,
                         "message": "UPDATE/DELETE без WHERE"})
    if re.match(r"\s*SELECT", sql, re.IGNORECASE) and \
       not re.search(r"\bLIMIT\b", sql, re.IGNORECASE) and \
       not re.search(r"\bCOUNT\(", sql, re.IGNORECASE):
        findings.append({"rule_id": "NO_PAGINATION", "risk_score": 4,
                         "message": "SELECT без LIMIT"})
    overall = max((f["risk_score"] for f in findings), default=0)
    return {"findings": findings, "overall_risk": overall, "approved": overall < 4.0}


section("Тест mock_judge")
for sql in ["SELECT * FROM clients", "SELECT id FROM clients LIMIT 100", "DELETE FROM clients"]:
    r = mock_judge(sql)
    print(f"\n SQL: {sql}")
    print(f"   approved: {r['approved']}, risk: {r['overall_risk']}")
    for f in r["findings"]:
        print(f"   - {f['rule_id']}: {f['message']}")


## 3. Reflector — переписывает findings → Lesson


In [ ]:
##
# @brief Reflector: из findings формирует lesson-ы.
# @details
#   В реальной системе reflector — это Qwen-7B вызов с promptom.
#   Здесь — словарь «rule_id → готовый текст урока». Концептуально
#   то же самое: structured-output из судьи.
LESSON_TEMPLATES = {
    "SELECT_STAR": Lesson("SELECT_STAR",
                          "Не используй SELECT *, перечисляй колонки явно.",
                          "SELECT * FROM clients",
                          "SELECT id, full_name FROM clients"),
    "DML_NO_WHERE": Lesson("DML_NO_WHERE",
                           "UPDATE/DELETE всегда требует WHERE с predicate по PK.",
                           "UPDATE clients SET balance=0",
                           "UPDATE clients SET balance=0 WHERE client_id=$1"),
    "NO_PAGINATION": Lesson("NO_PAGINATION",
                            "Любой SELECT должен иметь LIMIT (или keyset pagination).",
                            "SELECT id FROM clients ORDER BY ts DESC",
                            "SELECT id FROM clients ORDER BY ts DESC LIMIT 100"),
    "DIRECT_SENSITIVE": Lesson("DIRECT_SENSITIVE",
                               "Чувствительные поля маскируй (LEFT, hash).",
                               "SELECT passport FROM clients",
                               "SELECT LEFT(passport, 4) || '******' FROM clients"),
}


def reflector(findings, prev_reflection):
    """@brief Mock-reflector: lookup + дедуп."""
    new_lessons = []
    for f in findings:
        if f["rule_id"] in LESSON_TEMPLATES:
            new_lessons.append(LESSON_TEMPLATES[f["rule_id"]])
    # Дедуп по rule_id, окно 5 последних
    combined = prev_reflection + new_lessons
    seen = {}
    for l in combined:
        seen[l.rule_id] = l
    return list(seen.values())[-5:]


## 4. Полный цикл: generator → judge → reflector → loop


In [ ]:
##
# @brief Один полный прогон цикла генератор↔судья.
def run_loop(task, use_reflection, max_iter=5):
    state = LoopState(task=task)
    for it in range(1, max_iter + 1):
        state.iteration = it
        sql = mock_generator(state.task, state.reflection if use_reflection else [])
        state.sql_history.append(sql)
        audit = mock_judge(sql)
        state.audit_history.append(audit)
        if audit["approved"]:
            state.approved = True
            state.final_sql = sql
            return state
        # reflector — только если флаг включён
        if use_reflection:
            state.reflection = reflector(audit["findings"], state.reflection)
    state.final_sql = sql
    return state


def print_run(label, state):
    section(label)
    print(f"  approved:      {state.approved}")
    print(f"  iterations:    {state.iteration}")
    print(f"  final risk:    {state.audit_history[-1]['overall_risk']}")
    print(f"  reflection memory at end ({len(state.reflection)} lessons):")
    for l in state.reflection:
        print(f"    - {l}")
    print(f"  trajectory of risk:")
    for i, a in enumerate(state.audit_history, 1):
        print(f"    iter {i}: risk={a['overall_risk']}, findings={[f['rule_id'] for f in a['findings']]}")


section("A — БЕЗ reflection")
state_a = run_loop("покажи всех клиентов с балансом > 0", use_reflection=False)
print_run("Без reflection", state_a)

print()
section("B — С reflection")
state_b = run_loop("покажи всех клиентов с балансом > 0", use_reflection=True)
print_run("С reflection", state_b)


## 5. Главная метрика — «% задач, где vuln_class repeats»

Это **прямой ответ на исследовательский вопрос ТЗ**.


In [ ]:
def repeats_count(state):
    """@brief Сколько раз одно и то же правило срабатывало дважды."""
    seen = []
    repeats = 0
    for audit in state.audit_history:
        for f in audit["findings"]:
            if f["rule_id"] in seen:
                repeats += 1
            seen.append(f["rule_id"])
    return repeats


section("Сравнение метрик")
print(f"{'метрика':<28} {'без reflection':>18} {'с reflection':>18}")
print("-" * 64)
print(f"{'iterations_used':<28} {state_a.iteration:>18} {state_b.iteration:>18}")
print(f"{'approved':<28} {str(state_a.approved):>18} {str(state_b.approved):>18}")
print(f"{'final risk_score':<28} {state_a.audit_history[-1]['overall_risk']:>18} {state_b.audit_history[-1]['overall_risk']:>18}")
print(f"{'rules повторились':<28} {repeats_count(state_a):>18} {repeats_count(state_b):>18}")


## Итог

Мы увидели проблему **под микроскопом** и **симуляцию решения** из ADR.

## Куда дальше

- **Описание проблемы:** [problems/engineering/02-reflection-memory-loop/README.md](../../problems/engineering/02-reflection-memory-loop/README.md)
- **Варианты решения + почему так:** [problems/engineering/02-reflection-memory-loop/solutions.md](../../problems/engineering/02-reflection-memory-loop/solutions.md)
- **Архитектура цикла:** [docs/adr/0002-loop-architecture-langgraph.md](../../docs/adr/0002-loop-architecture-langgraph.md)
